# Librairies

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")  # racine du projet

import matplotlib.pyplot as plt
import pandas as pd

# Données

In [ ]:
from crop_recommendation_tp.config import load_config

config = load_config()
df = pd.read_csv(config["dataset"]["path"])
df.shape

(2200, 8)

## Validation des données

In [ ]:
from crop_recommendation_tp.config import load_yaml
from crop_recommendation_tp.validation import validate

schema = load_yaml(config["dataset"]["schema"])
errors = validate(df, schema)
errors

[]

# Fractionnement des données

In [4]:
from sklearn.model_selection import train_test_split

target = config["dataset"]["target"]
features = [c for c in schema["columns"] if c != target]

X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df[target],
    test_size=config["split"]["test_size"],
    random_state=config["split"]["random_seed"],
    stratify=df[target],
)
X_train.shape, X_test.shape

((1760, 7), (440, 7))

# MLFLOW

In [5]:
import mlflow

mlflow.set_tracking_uri(config["mlflow"]["tracking_uri"])
mlflow.set_experiment(config["mlflow"]["experiment_name"])

/Users/bertrandfillion/Python Projects/mlflow/crop_recommendation_tp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/11/03 09:22:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/03 09:22:02 INFO mlflow.store.db.utils: Updating database tables
2026/11/03 09:22:02 INFO mlflow.tracking.fluent: Experiment with name 'crop-recommendation' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/bertrandfillion/Python Projects/mlflow/crop_recommendation_tp/mlruns/1', creation_time=1793694122782, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1793694122782, lifecycle_stage='active', name='crop-recommendation', tags={}, trace_location=None, workspace='default'>

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)

In [ ]:
params = {
    "n_estimators": 100,
    "random_state": config["split"]["random_seed"],
}

with mlflow.start_run(run_name="random_forest_base"):
    model = RandomForestClassifier(**params).fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    f1_macro = f1_score(y_test, predictions, average="macro")
    mlflow.log_params(params)
    mlflow.log_metrics({"accuracy": accuracy, "f1_macro": f1_macro})
    mlflow.sklearn.log_model(
        model, name="model", skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

accuracy, f1_macro

2026/11/03 09:22:09 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/bertrandfillion/Python Projects/mlflow/crop_recommendation_tp
2026/11/03 09:22:13 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/bertrandfillion/Python Projects/mlflow/crop_recommendation_tp
2026/11/03 09:22:13 INFO mlflow.utils.environment: Detected uv project at /Users/bertrandfillion/Python Projects/mlflow/crop_recommendation_tp. Attempting to export requirements via 'uv export'.
2026/11/03 09:22:13 INFO mlflow.utils.uv_utils: Exported 155 dependencies via uv
2026/11/03 09:22:13 INFO mlflow.utils.environment: Successfully exported 155 requirements from uv project. Skipping package capture based inference.
2026/11/03 09:22:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


(0.9954545454545455, 0.9954517027687758)

In [14]:
def train_and_log(run_name, params):
    with mlflow.start_run(run_name=run_name):
        model = RandomForestClassifier(**params).fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        f1_macro = f1_score(y_test, predictions, average="macro")
        report = classification_report(y_test, predictions)
        mlflow.log_text(report, "classification_report.txt")
        print(report)
        fig, ax = plt.subplots(figsize=(10, 10))
        ConfusionMatrixDisplay.from_predictions(
            y_test, predictions, xticks_rotation=90, colorbar=False, cmap="Blues", ax=ax
        )
        mlflow.log_figure(fig, "confusion_matrix.png")
        plt.close(fig)
        mlflow.log_params(params)
        mlflow.log_metrics({"accuracy": accuracy, "f1_macro": f1_macro})
        mlflow.sklearn.log_model(
            model,
            name="model",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
    return accuracy, f1_macro

In [15]:
seed = config["split"]["random_seed"]
train_and_log(
    "random_forest_peu_profonde",
    {"n_estimators": 20, "max_depth": 3, "random_state": seed},
)

/Users/bertrandfillion/Python Projects/mlflow/crop_recommendation_tp/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/bertrandfillion/Python Projects/mlflow/crop_recommendation_tp/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/bertrandfillion/Python Projects/mlflow/crop_recommendation_tp/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with 

              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       0.46      0.85      0.60        20
    chickpea       1.00      1.00      1.00        20
     coconut       0.87      1.00      0.93        20
      coffee       1.00      0.95      0.97        20
      cotton       0.77      1.00      0.87        20
      grapes       1.00      1.00      1.00        20
        jute       0.00      0.00      0.00        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       0.47      1.00      0.63        20
       maize       0.88      0.70      0.78        20
       mango       1.00      0.90      0.95        20
   mothbeans       0.00      0.00      0.00        20
    mungbean       0.95      1.00      0.98        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00      1.00        20
      papaya       1.00    

2026/11/03 09:33:53 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/bertrandfillion/Python Projects/mlflow/crop_recommendation_tp
2026/11/03 09:33:53 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/bertrandfillion/Python Projects/mlflow/crop_recommendation_tp
2026/11/03 09:33:53 INFO mlflow.utils.environment: Detected uv project at /Users/bertrandfillion/Python Projects/mlflow/crop_recommendation_tp. Attempting to export requirements via 'uv export'.
2026/11/03 09:33:53 INFO mlflow.utils.uv_utils: Exported 155 dependencies via uv
2026/11/03 09:33:53 INFO mlflow.utils.environment: Successfully exported 155 requirements from uv project. Skipping package capture based inference.
2026/11/03 09:33:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


(0.8272727272727273, 0.7837217674477599)